In [ ]:
import json
import boto3


BUCKET = "kafka-spark-stock-project-kris"

PREFIX = (
    "stock-market/bronze/rest/basic_financials/"
)


s3 = boto3.client("s3")


# Find JSON files in the Basic Financials Bronze path
response = s3.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX,
)

files = [
    obj["Key"]
    for obj in response.get("Contents", [])
    if obj["Key"].endswith(".json")
]

if not files:
    raise RuntimeError(
        "No Basic Financials JSON files found."
    )


# Use the newest file
latest_file = sorted(files)[-1]

print(f"Reading:\ns3://{BUCKET}/{latest_file}\n")


obj = s3.get_object(
    Bucket=BUCKET,
    Key=latest_file,
)

payload = json.loads(
    obj["Body"].read().decode("utf-8")
)


for record in payload["records"]:

    symbol = record["symbol"]
    metrics = record.get("metrics", {})

    print("=" * 70)
    print(
        f"{symbol} | "
        f"{len(metrics)} available metrics"
    )
    print("=" * 70)

    for metric_name in sorted(metrics.keys()):
        print(
            f"{metric_name}: "
            f"{metrics[metric_name]}"
        )

    print()